# CSI2PointCloud — TEST (métricas ICP + visualización)

Evalúa el modelo entrenado sobre el **test reservado** (último 15% cronológico de cada acción,
que el modelo nunca vio). Carga el modelo y el `split.json` que generó **entrenar.ipynb**.

- **ICP fitness** / **ICP RMSE** (en metros) y **Chamfer** bidireccional.
- Visualización GT vs predicho con los **colores de Määttä** (viridis por distancia).

> Requiere haber corrido **entrenar.ipynb** antes (deja `modelo/csi2pc_v2.pth` y `split.json`).

## 1. Configuración, dataset y carga del modelo

In [ ]:
import os, sys, glob, json
import numpy as np, scipy.io as scio, torch
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

HERE = os.path.abspath('')
REPO = os.environ.get('CSI2PC_REPO', os.path.join(HERE, '..', 'csi2pointcloud'))
DATA = os.environ.get('CSI2PC_DATA', os.path.join(HERE, '..', 'mmfi_dataset'))
OUT  = os.environ.get('CSI2PC_OUT',  os.path.join(HERE, 'resultados'))
sys.path.append(REPO)
from src.csi2pointcloud.csi2pointcloud_arch01 import CSI2PointCloudModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# items en el mismo orden que en el entrenamiento
items, actions = [], []
for mat in sorted(glob.glob(os.path.join(DATA, '**', 'wifi-csi', '*.mat'), recursive=True)):
    b = mat.replace('wifi-csi', 'lidar').replace('.mat', '.bin')
    if os.path.exists(b):
        items.append((mat, b)); actions.append(os.path.basename(os.path.dirname(os.path.dirname(mat))))

test_idx = json.load(open(os.path.join(OUT, 'split.json')))['test']

model = CSI2PointCloudModel(256, 4, 3, 3, 1200, 3, 114, 10).to(device)
model.load_state_dict(torch.load(os.path.join(OUT, 'modelo', 'csi2pc_v2.pth'), map_location=device)['model_state_dict'])
model.eval()
print('Modelo cargado. Frames de test:', len(test_idx), '| device', device)

def csi_tensor(mat):
    m = scio.loadmat(mat)
    amp = m['CSIamp'].astype(np.float64); pha = m['CSIphase'].astype(np.float64)
    amp[np.isinf(amp)] = np.nan; pha[np.isinf(pha)] = np.nan
    if np.any(np.isnan(amp)): amp[np.isnan(amp)] = np.nanmean(amp)
    if np.any(np.isnan(pha)): pha[np.isnan(pha)] = np.nanmean(pha)
    amp = (amp - np.nanmin(amp)) / (np.nanmax(amp) - np.nanmin(amp) + 1e-9)
    csi = np.stack([amp, pha], axis=2).astype(np.float32)
    x = torch.from_numpy(csi); return (x - x.mean()) / (x.std() + 1e-9)

## 2. Métricas ICP sobre el TEST (fitness, RMSE, Chamfer)

In [ ]:
THR = 0.10  # umbral de correspondencia en metros

def nn_dist(A, B):
    kdt = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(B)))
    d = np.empty(len(A))
    for i, p in enumerate(A):
        _, _, d2 = kdt.search_knn_vector_3d(p, 1); d[i] = np.sqrt(d2[0])
    return d

por_accion = {}
fits, rmses, chamfs = [], [], []
for i in test_idx:
    gt = np.frombuffer(open(items[i][1], 'rb').read(), dtype=np.float64).reshape(-1,3)
    with torch.no_grad():
        pred, _ = model(csi_tensor(items[i][0]).unsqueeze(0).to(device))
    pred = pred.squeeze(0).cpu().numpy().astype(np.float64)
    pred_m = pred * gt.std() + gt.mean()
    reg = o3d.pipelines.registration.registration_icp(
        o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pred_m)),
        o3d.geometry.PointCloud(o3d.utility.Vector3dVector(gt)),
        THR, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    ch = nn_dist(pred_m, gt).mean() + nn_dist(gt, pred_m).mean()
    fits.append(reg.fitness); rmses.append(reg.inlier_rmse); chamfs.append(ch)
    por_accion.setdefault(actions[i], []).append((reg.fitness, reg.inlier_rmse, ch))

print('TEST (%d frames, umbral %.2f m)' % (len(fits), THR))
print('  ICP fitness : %.4f' % np.mean(fits))
print('  ICP RMSE    : %.4f m' % np.mean(rmses))
print('  Chamfer     : %.4f m' % np.mean(chamfs))
print('-'*48)
for a in sorted(por_accion):
    v = np.array(por_accion[a])
    print('  %s | fitness %.3f | RMSE %.4f m | Chamfer %.4f m | n=%d' % (a, v[:,0].mean(), v[:,1].mean(), v[:,2].mean(), len(v)))

## 3. Visualización GT vs Predicho (colores de Määttä)

In [ ]:
def plot_maata(ax, P, titulo):
    h, prof, alt = P[:,0], P[:,2], -P[:,1]
    dist = np.linalg.norm(P, axis=1)
    ax.scatter(h, prof, alt, c=dist, cmap='viridis', s=7, alpha=0.9)
    ax.set_title(titulo); ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('altura (-Y)')
    ax.view_init(elev=12, azim=-75)

i = test_idx[len(test_idx)//2]  # un frame del test (cambia el indice para ver otros)
gt = np.frombuffer(open(items[i][1], 'rb').read(), dtype=np.float64).reshape(-1,3)
gt = (gt - gt.mean()) / (gt.std() + 1e-9)
with torch.no_grad():
    pred, _ = model(csi_tensor(items[i][0]).unsqueeze(0).to(device))
pred = pred.squeeze(0).cpu().numpy()

fig = plt.figure(figsize=(13,6))
plot_maata(fig.add_subplot(1,2,1, projection='3d'), gt,   'Ground-truth (Kinect)  [%s]' % actions[i])
plot_maata(fig.add_subplot(1,2,2, projection='3d'), pred, 'Predicho (CSI -> IA)')
plt.suptitle('Generacion de nube de puntos - colores de Maata (viridis por distancia)')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'pred_vs_gt.png'), dpi=120, bbox_inches='tight')
plt.show()